# Nemotron TranscribeをT4で試す

このノートブックは、NeMo-Speech.cppの実験的な小GPU経路で日本語音声を文字起こしします。最初に公開コードとモデルの固定リビジョンを取得し、T4上でビルド・変換してから音声を処理します。

## 0. 実行前に確認する

- Colabのランタイムを **T4 GPU / High RAM** にします。必要なRAMは約50 GiB以上、空きディスクは180 GB以上です。
- 変換中はBF16モデル約67 GB、変換用データ約66 GB、量子化出力約22 GBを使います。処理後もモデル重みは `/content` に残り、ZIPには入りません。
- 既定の `GPU_LAYERS = 0` はdecoderをCPUで実行し、音声encoderはGPUを使います。decoderは実験的なQ4_0 experts / Q8_0 dense構成です。GPUには約4 GiB以上の空きが必要です。4 GiB GPUでの動作を保証する設定ではありません。
- 初回のセットアップはモデル取得、ビルド、変換を行うため時間がかかります。各段階のログは `/content/nemotron-transcribe` に保存します。

In [ ]:
#@title 実行設定
from pathlib import Path

REPO_URL = "https://github.com/Sunwood-ai-labs/nemotron-transcribe-colab-lab.git"
CODE_REVISION = "33a38c8faaed84eb8a4c781f35327d2c21a9e2f2"  # 固定コミットSHA
REPO_ROOT = Path("/content/nemotron-transcribe-colab-lab")
GPU_LAYERS = 0  #@param [0, 8] {type:"integer"}
USE_UPLOAD = False  #@param {type:"boolean"}
DOWNLOAD_REPORT = False  #@param {type:"boolean"}

if GPU_LAYERS not in (0, 8):
    raise ValueError("GPU_LAYERSは0または8にしてください。")
print(f"取得元: {REPO_URL}\nコード指定: {CODE_REVISION}\ndecoder GPU layers: {GPU_LAYERS}")

## 1. 公開コードを固定して取得する

このノートブックは固定コミットから scripts/lab.py を読み込みます。取得したSHAは下に表示され、結果ZIPにも保存されます。

In [ ]:
import re
import subprocess
import sys

if not re.fullmatch(r"main|[0-9a-f]{40}", CODE_REVISION):
    raise ValueError("CODE_REVISIONはmainまたは40桁のコミットSHAにしてください。")
if REPO_ROOT.exists() and not (REPO_ROOT / ".git").exists():
    raise RuntimeError(f"保存先がGitリポジトリではありません: {REPO_ROOT}")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_ROOT)], check=True)
remote = subprocess.check_output(["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"], text=True).strip()
if remote.removesuffix(".git").lower() != REPO_URL.removesuffix(".git").lower():
    raise RuntimeError(f"保存先のoriginが想定と異なります: {remote}")
if subprocess.check_output(["git", "-C", str(REPO_ROOT), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("取得先に未保存の変更があります。内容を退避してから再実行してください。")
subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", CODE_REVISION], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip().lower()
if re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION) and SOURCE_COMMIT != CODE_REVISION:
    raise RuntimeError(f"取得したSHAが一致しません: {SOURCE_COMMIT}")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import lab
print(f"取得したコードSHA: {SOURCE_COMMIT}")

## 2. ランタイムを確認する

この確認はPyTorchをimportせず、GPU・RAM・空き容量・`nvcc`・既存PyTorchの版を読みます。条件を満たさない場合は、数十GBのモデルを取得する前に停止します。

In [ ]:
from IPython.display import display

runtime = lab.check_runtime()
display(runtime)

## 3. モデルと実行ファイルを準備する

この段階で依存関係を整え、BF16モデルを取得し、T4向けネイティブ実行ファイルをビルドしてQ4_0 / Q8_0に変換します。ログは保存しながら、セルには節目と簡潔な進捗だけを表示します。

In [ ]:
prepared = lab.prepare()
display({key: value for key, value in prepared.items() if key != "runtime"})

## 4. サンプル音声を聞く

同梱のGoogle FLEURS日本語音声と参照文です。各音声はここで一度だけ再生できます。次のセルはリポジトリ内に保存したT4の実行結果を読みます。今回の実行結果とは分けて表示します。

In [ ]:
from IPython.display import Audio, display
import soundfile

for sample in lab.SAMPLES:
    print(f"ID {sample['id']} / 参照文: {sample['reference']}")
    audio, sample_rate = soundfile.read(str(sample["path"]), dtype="float32")
    display(Audio(audio, rate=sample_rate, autoplay=False))

### 前回保存済みのT4・Q4_0 / CPU併用の出力

元データは [保存済み結果JSON](evidence/2026-10-30/small-gpu-results.json) です。T4上でdecoderのGPU layer数を0にした保存済み出力だけを表示します。4 GiBの空きVRAMを残すよう同じT4上でメモリを確保した実験で、4 GiB GPUでの実行ではありません。2音声の参考例で、今回の結果や一般的な日本語精度を示すものではありません。

In [ ]:
import json

saved_path = REPO_ROOT / "evidence" / "2026-10-30" / "small-gpu-results.json"
saved_evidence = json.loads(saved_path.read_text(encoding="utf-8"))
saved_runs = [run for run in saved_evidence["runs"] if run["gpu_layers"] == 0]
for run in saved_runs:
    transcript = run["transcript"].removesuffix("<|im_end|>").strip()
    print(f"ID {run['sample_id']} / T4保存済み出力: {transcript}")

## 5. 今回の音声を文字起こしする

既定では2件を `GPU_LAYERS` の値で1回ずつ処理します。各出力に音声SHA256、設定、経過秒、ログの保存先を記録します。量子化は実験的な経路なので、出力を正解文と見比べてください。

In [ ]:
RUNS = lab.compare_samples(gpu_layers=GPU_LAYERS)
for run in RUNS:
    print(f"ID {Path(run['audio']).name} / decoder GPU layers={run['gpu_layers']}")
    print(run["transcript"])
    print(f"経過時間: {run['elapsed_seconds_including_model_load']:.1f}秒 / JSON: {run['result_json']}\n")

## 6. 自分のWAVを追加する（任意）

アップロードは初期設定で無効です。使う場合は設定セルの `USE_UPLOAD = True` に変更して、このセルを実行してください。入力はモノラルWAV（PCMまたはfloat）です。

In [ ]:
if USE_UPLOAD:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        upload_path = Path("/content") / next(iter(uploaded))
        RUNS.extend([lab.transcribe(upload_path, gpu_layers=GPU_LAYERS)])
        print(RUNS[-1]["transcript"])
else:
    print("アップロードは無効です。設定セルでUSE_UPLOAD=Trueにすると選択できます。")

## 7. 結果とログを書き出す

ZIPには実行レポートと必要なログだけを含めます。モデル重みと入力音声は含みません。Colab CLIや自動実行ではダウンロードUIを開かず、設定セルの `DOWNLOAD_REPORT = True` を選んだ場合だけブラウザーのダウンロードを呼び出します。

In [ ]:
report_path = lab.export_results(RUNS, source_commit=SOURCE_COMMIT)
print(f"保存したレポート: {report_path}")
from IPython.display import FileLink, display
display(FileLink(str(report_path)))
if DOWNLOAD_REPORT:
    from google.colab import files

    files.download(str(report_path))